# 🩺 EDA — Athlete Injury Risk Detection

Data exploration:

1. **Audit of the 4 candidate Kaggle datasets** (size, target, balance, temporal dimension)
2. **SIRP-600 focus** (retained real dataset): distributions & class imbalance
3. **Synthetic dataset**: injury events, ACWR, load zones, time series
4. **Correlations** & conclusions

> Key finding: no real dataset has a daily time series → the ACWR, and a genuinely
> predictive injury target, are only possible on the synthetic dataset (dual-track strategy).

> Sections 1–2 need the raw Kaggle datasets (gitignored); they skip cleanly when those are
> absent, so this notebook runs end to end in CI. The synthetic sections always run.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.width', 200)
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'

# The raw Kaggle datasets are gitignored (private). This flag lets the notebook run
# end to end in CI, where they are absent: the audit and SIRP-600 sections skip
# cleanly, while the synthetic sections — generated on the fly — always run.
SIRP_FILE = RAW / 'sirp-600' / 'High_Accuracy_Sport_Injury_Dataset.xlsx'
HAS_REAL_DATA = SIRP_FILE.exists()
if not HAS_REAL_DATA:
    print('Real Kaggle datasets not present — the audit & SIRP-600 sections will be skipped.')

## 1. Audit of the candidate datasets

In [ ]:
if HAS_REAL_DATA:
    mrsimple = pd.read_csv(RAW / 'injury-prediction-mrsimple07' / 'injury_data.csv')
    univ = pd.read_csv(RAW / 'university-football-injury' / 'data.csv')
    epl = pd.read_csv(RAW / 'epl-player-injuries' / 'player_injuries_impact.csv')
    sirp = pd.read_excel(SIRP_FILE)

    audit = pd.DataFrame([
        {'dataset': 'mrsimple07', 'rows': len(mrsimple), 'columns': mrsimple.shape[1],
         'target': 'Likelihood_of_Injury',
         'balance': mrsimple['Likelihood_of_Injury'].value_counts(normalize=True).round(2).to_dict(),
         'temporal': 'no (snapshot)'},
        {'dataset': 'university-football', 'rows': len(univ), 'columns': univ.shape[1],
         'target': 'Injury_Next_Season',
         'balance': univ['Injury_Next_Season'].value_counts(normalize=True).round(2).to_dict(),
         'temporal': 'no (snapshot)'},
        {'dataset': 'epl-injuries', 'rows': len(epl), 'columns': epl.shape[1],
         'target': 'none (match impact)', 'balance': '-', 'temporal': 'match history'},
        {'dataset': 'sirp-600', 'rows': len(sirp), 'columns': sirp.shape[1],
         'target': 'Injury_Risk',
         'balance': sirp['Injury_Risk'].value_counts(normalize=True).round(2).to_dict(),
         'temporal': 'no (snapshot)'},
    ])
    display(audit)
else:
    print('Skipped — raw Kaggle datasets not available.')

**Reading:** `mrsimple07` and `university-football` have an **exact 50/50** balance (suspicious, flagged as potentially artificial in the literature). `epl` has no usable risk target. **SIRP-600** is the only one with a *natural* imbalance (~68/32) → retained as the real dataset.

## 2. SIRP-600 focus (retained real dataset)

In [ ]:
if HAS_REAL_DATA:
    from injury_risk.data.load_dataset import load_sirp600, SIRP_TARGET, SIRP_FEATURE_COLS
    sirp = load_sirp600()
    print(sirp.shape)
    display(sirp.describe().round(2))
else:
    print('Skipped — SIRP-600 not available.')

In [ ]:
if HAS_REAL_DATA:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sirp[SIRP_TARGET].value_counts().sort_index().plot(
        kind='bar', ax=axes[0], color=['#22c55e', '#ef4444'])
    axes[0].set_title('SIRP-600 — class imbalance (justifies SMOTE)')
    axes[0].set_xlabel('Injury_Risk (0=low, 1=risk)')
    axes[0].set_ylabel('count')

    sns.boxplot(data=sirp, x=SIRP_TARGET, y='Training_Intensity', ax=axes[1],
                palette=['#22c55e', '#ef4444'])
    axes[1].set_title('Training intensity vs risk')
    plt.tight_layout(); plt.show()
else:
    print('Skipped — SIRP-600 not available.')

## 3. Synthetic dataset — injury events, ACWR & time series

The generator simulates **actual injury events** (a discrete-time logistic hazard driven
by the latent risk), not a discretised score. The model target is `injury_next_7d`: will
the athlete get injured within the next 7 days, given everything known up to today.

In [ ]:
from injury_risk.data.generate_synthetic import generate, DEFAULT_OUTPUT
from injury_risk.features.engineering import build_features

if DEFAULT_OUTPUT.exists():
    synth = pd.read_parquet(DEFAULT_OUTPUT)
else:
    synth = generate()
synth = build_features(synth)
print(synth.shape)
synth[['athlete_id', 'date', 'training_load', 'acwr', 'acwr_zone',
       'soreness', 'is_injured', 'injury_onset', 'injury_next_7d']].head()

In [ ]:
# The injury target is rare and imbalanced — which is what justifies SMOTE.
# Modellable rows only: not already injured, and the 7-day horizon fully observed.
from injury_risk.config import TARGET_COL, WARMUP_DAYS

modellable = synth[(synth['day'] >= WARMUP_DAYS) & (~synth['is_injured']) & synth['horizon_complete']]
rate = modellable[TARGET_COL].mean()

print(f'injuries simulated   : {int(synth["injury_onset"].sum())} '
      f'({synth["injury_onset"].sum() / synth["athlete_id"].nunique():.1f} per athlete)')
print(f'days sidelined       : {synth["is_injured"].mean():.1%} of all athlete-days')
print(f'positive rate (7d)   : {rate:.2%} of modellable rows')

ax = pd.Series({'no injury (<7d)': 1 - rate, 'injury within 7d': rate}).plot(
    kind='bar', color=['#22c55e', '#ef4444'], figsize=(6, 4))
ax.set_title('Synthetic target — injury within 7 days (imbalanced)')
ax.set_ylabel('proportion'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
# ACWR distribution + business zones
stable = synth[synth['day'] >= 28]
fig, ax = plt.subplots(figsize=(9,4))
ax.hist(stable['acwr'], bins=60, color='#3b82f6', alpha=0.7)
for x, c, lbl in [(0.8,'#22c55e','optimal 0.8'),(1.3,'#f59e0b','elevated 1.3'),(1.5,'#ef4444','danger 1.5')]:
    ax.axvline(x, color=c, linestyle='--', label=lbl)
ax.set_title('ACWR distribution (danger zone > 1.5)')
ax.set_xlabel('ACWR'); ax.legend(); ax.set_xlim(0, 2.5)
plt.tight_layout(); plt.show()
stable['acwr_zone'].value_counts(normalize=True).round(3)

In [ ]:
# One athlete's time series: load, 7/28-day rolling and ACWR
a = synth[synth['athlete_id'] == 0].sort_values('day')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(a['day'], a['training_load'], color='lightgray', label='raw load')
ax1.plot(a['day'], a['training_load_7d'], color='#2563eb', label='7-day mean (acute)')
ax1.plot(a['day'], a['training_load_28d'], color='#dc2626', label='28-day mean (chronic)')
ax1.set_title('Athlete #0 — load & moving averages'); ax1.legend(); ax1.set_ylabel('load')
ax2.plot(a['day'], a['acwr'], color='#7c3aed')
ax2.axhspan(0.8, 1.3, color='#22c55e', alpha=0.15)
ax2.axhline(1.5, color='#ef4444', linestyle='--', label='danger')
ax2.set_title('ACWR over the season'); ax2.set_xlabel('day'); ax2.set_ylabel('ACWR'); ax2.legend()
plt.tight_layout(); plt.show()

## 4. Correlations (synthetic dataset)

In [ ]:
from injury_risk.features.engineering import SYNTHETIC_FEATURE_COLS

# Correlate the features with the observed injury target, on modellable rows.
corr = modellable[SYNTHETIC_FEATURE_COLS + [TARGET_COL]].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title(f'Correlation matrix (features vs {TARGET_COL})')
plt.tight_layout(); plt.show()

corr[TARGET_COL].drop(TARGET_COL).sort_values(ascending=False).round(3)

## Conclusions

- The injury target is **rare** (~5% of modellable days), which is what justifies SMOTE.
- Linear correlations with a single-day snapshot of the features are **weak** — injury is
  a stochastic event, not a deterministic function of today's numbers. The signal lives in
  the *combination* of features over time, which is what the model learns (and SHAP then
  attributes): see the results in the README.
- Only the synthetic dataset carries a daily time series, so it is the only one on which a
  temporal ACWR — and this predictive target — can exist. That is the dual-track rationale.

➡️ Next: feature engineering (`src/injury_risk/features/`), models (`src/injury_risk/models/`),
SHAP explainability (`src/injury_risk/visualization/`).